In [6]:
import sqlite3
import pandas as pd

print("1. Đang khởi tạo CSDL SQLite...")
conn = sqlite3.connect('metro_madrid.db')
cursor = conn.cursor()

print("2. Đang đọc dữ liệu sạch...")
stops = pd.read_csv('processed_data/cleaned_stops.csv')
stop_times = pd.read_csv('processed_data/cleaned_stop_times.csv')

print("3. Đang đẩy dữ liệu vào các Bảng...")
# Bảng Tram (Stations)
stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].to_sql('Tram', conn, if_exists='replace', index=False)

# Xử lý Bảng Ket_Noi (Edges) - Tìm ga kế tiếp của mỗi chuyến tàu
stop_times = stop_times.sort_values(['trip_id', 'stop_sequence'])
stop_times['next_stop_id'] = stop_times.groupby('trip_id')['stop_id'].shift(-1)
stop_times['next_arrival'] = stop_times.groupby('trip_id')['arrival_time'].shift(-1)

# Tính thời gian di chuyển (travel_time) bằng giây
stop_times['travel_time'] = (pd.to_timedelta(stop_times['next_arrival']) - 
                             pd.to_timedelta(stop_times['departure_time'])).dt.total_seconds()

# Lọc bỏ các điểm cuối của hành trình
connections = stop_times.dropna(subset=['next_stop_id', 'travel_time'])

# Lưu vào bảng Ket_Noi
connections[['trip_id', 'stop_id', 'next_stop_id', 'travel_time']].to_sql('Ket_Noi', conn, if_exists='replace', index=False)

print("4. Đang tạo Index để C++ truy vấn siêu tốc...")
cursor.execute('CREATE INDEX idx_stop ON Ket_Noi(stop_id)')
cursor.execute('CREATE INDEX idx_next_stop ON Ket_Noi(next_stop_id)')

conn.commit()
conn.close()
print("=> HOÀN TẤT! File 'metro_madrid.db' đã được tạo.")

1. Đang khởi tạo CSDL SQLite...
2. Đang đọc dữ liệu sạch...
3. Đang đẩy dữ liệu vào các Bảng...
4. Đang tạo Index để C++ truy vấn siêu tốc...
=> HOÀN TẤT! File 'metro_madrid.db' đã được tạo.


In [7]:
# Mã hóa Trạm và Cạnh
import sqlite3
import pandas as pd

print("1. Đang mở CSDL Đồ thị...")
conn = sqlite3.connect('metro_madrid.db')

# Đọc 2 bảng lên
tram_df = pd.read_sql('SELECT * FROM Tram', conn)
ket_noi_df = pd.read_sql('SELECT * FROM Ket_Noi', conn)

print("2. Đang thực hiện Mã hóa Trạm (Nodes)...")
# Tạo bộ từ điển map: Mã chữ -> Số nguyên (0, 1, 2...)
stop_id_to_int = {stop_id: idx for idx, stop_id in enumerate(tram_df['stop_id'].unique())}

# Thêm cột định danh số nguyên 'node_id' vào bảng Tram
tram_df['node_id'] = tram_df['stop_id'].map(stop_id_to_int)

print("3. Đang thực hiện Mã hóa Cạnh (Edges)...")
# Thêm cột 'u' (đỉnh nguồn) và 'v' (đỉnh đích) bằng số nguyên vào bảng Ket_Noi
ket_noi_df['u'] = ket_noi_df['stop_id'].map(stop_id_to_int)
ket_noi_df['v'] = ket_noi_df['next_stop_id'].map(stop_id_to_int)

print("4. Đang cập nhật lại Database...")
# Ghi đè lại 2 bảng đã được mã hóa vào file .db
tram_df.to_sql('Tram', conn, if_exists='replace', index=False)
ket_noi_df.to_sql('Ket_Noi', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print("=> HOÀN TẤT MÃ HÓA! Cơ sở dữ liệu đồ thị đã sẵn sàng 100%.")

1. Đang mở CSDL Đồ thị...
2. Đang thực hiện Mã hóa Trạm (Nodes)...
3. Đang thực hiện Mã hóa Cạnh (Edges)...
4. Đang cập nhật lại Database...
=> HOÀN TẤT MÃ HÓA! Cơ sở dữ liệu đồ thị đã sẵn sàng 100%.
